# Module 19: Interactive Production Observability & Prometheus Metrics

### What You Will Discover
By running this notebook, you will explore Prometheus telemetry exposition, instrument Python code with Counters, Gauges, and Histograms, and inspect raw metrics format.

**Key Question Answered:** *Why should you never use high-cardinality labels (like user IDs or timestamps) in Prometheus metrics?*


In [ ]:
# Step 1: Instrumenting with prometheus_client
from prometheus_client import CollectorRegistry, Counter, Gauge, Histogram, generate_latest

registry = CollectorRegistry()
HTTP_REQUESTS = Counter(
    'sample_http_requests_total',
    'Total HTTP requests processed',
    ['endpoint', 'status'],
    registry=registry
)


In [ ]:
# Step 2: Incrementing metric counters
HTTP_REQUESTS.labels(endpoint='/api/items', status='200').inc()
HTTP_REQUESTS.labels(endpoint='/api/items', status='200').inc()
HTTP_REQUESTS.labels(endpoint='/api/login', status='401').inc()
print('Metrics counters incremented.')


In [ ]:
# Step 3: Generating and inspecting Prometheus exposition format
metrics_text = generate_latest(registry).decode('utf-8')
for line in metrics_text.splitlines()[:6]:
    print(line)


### 🔮 Prediction Prompt
**Before running the next cell:** If you have 100,000 users and you add `user_id` as a Prometheus label, what will happen to the memory consumption of your application and Prometheus server? Write down your prediction.


In [ ]:
# Surprising Result: High Cardinality Memory Explosion
print('Metric Cardinality = (Endpoints) x (Status Codes) x (User IDs)')
print('With user_id label: 10 endpoints x 5 statuses x 100,000 users = 5,000,000 time series!')
print('Explanation: High cardinality creates millions of time series in memory, crashing Prometheus!')
print('Rule: Only use bounded, finite enum labels in metrics. Put user IDs in structured logs!')


### Latency Histograms for Percentiles (p50, p95, p99)
Histograms track latency distributions in discrete buckets.


In [ ]:
LATENCY = Histogram(
    'request_duration_seconds',
    'Request latency distribution',
    ['endpoint'],
    buckets=[0.01, 0.05, 0.1, 0.5, 1.0],
    registry=registry
)

LATENCY.labels(endpoint='/api/items').observe(0.034)
LATENCY.labels(endpoint='/api/items').observe(0.082)
print('Observed latencies in histogram buckets.')


### Gauges for Dynamic Instantaneous Values
Gauges can increase and decrease (e.g. active worker pool threads or queue size).


In [ ]:
ACTIVE_WORKERS = Gauge('active_workers_count', 'Number of active worker threads', registry=registry)
ACTIVE_WORKERS.set(8)
ACTIVE_WORKERS.dec(2)
print('Gauge value adjusted to reflect active pool state.')


### 🛠️ Interactive Challenge: Fix the Cardinality Explosion Bug
The following function accidentally includes a unique transaction UUID as a label in a Prometheus metric. Fix it by removing the high-cardinality label.


In [ ]:
# TODO: FIX ME - Remove high-cardinality transaction_id label from Prometheus metric
SAFE_ORDERS = Counter('orders_total', 'Total orders', ['status'], registry=registry)

def record_order(status: str, tx_id: str):
    # FIX: SAFE_ORDERS.labels(status=status).inc()
    SAFE_ORDERS.labels(status=status).inc()
    # Transaction ID belongs in structured logs, not metrics!
    return f'Logged tx {tx_id}'

record_order('success', 'tx_998124_uuid')
print('Safe metric recorded without cardinality explosion!')


### 🏁 Summary & Next Steps
- Use Counters for cumulative counts; Histograms for latencies; Gauges for state.
- Avoid high-cardinality labels (user IDs, timestamps, UUIDs).
- Use multi-stage Docker builds with unprivileged non-root users.
- Run `python 01_prometheus_metrics_demo.py` and review `02_multistage_dockerfile_demo.md`.
- Follow [PROJECT_GUIDE.md](PROJECT_GUIDE.md) to implement the container observability pipeline.
